In [1]:
import numpy as np
import pandas as pd
import os

# Load raw customer data (Bronze)

In [2]:
!git clone https://github.com/vinculum3141-ship-it/utilus_home_assignment.git

Cloning into 'utilus_home_assignment'...
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 88 (delta 6), reused 86 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (88/88), 132.05 KiB | 13.20 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [3]:
base_path = '/content/utilus_home_assignment/assignment/'

In [4]:
cust_raw = pd.read_csv(os.path.join(base_path, "customers.csv"))
print("Customer data loaded successfully. All rows are kept.")
print(f"Total rows in subs_raw: {len(cust_raw)}")
print(f"Number of unique customer_ids: {cust_raw['customer_id'].nunique()}")
print("Subscriptions data loaded successfully. First 5 rows:")
display(cust_raw.head())

Customer data loaded successfully. All rows are kept.
Total rows in subs_raw: 40
Number of unique customer_ids: 39
Subscriptions data loaded successfully. First 5 rows:


,customer_id,signup_date,country
0,C001,2024-01-02,NL
1,C002,2024-01-11,DE
2,C003,2024-01-27,FR
3,C004,2024-02-05,NL
4,C005,2024-02-28,SE


# Clean and validate customer data (Silver)

In [17]:
cust_silver = cust_raw.copy()
print(f"Silver DataFrame 'cust_silver' created with {len(cust_silver)} rows.")
display(cust_silver.head())

Silver DataFrame 'cust_silver' created with 40 rows.


,customer_id,signup_date,country
0,C001,2024-01-02,NL
1,C002,2024-01-11,DE
2,C003,2024-01-27,FR
3,C004,2024-02-05,NL
4,C005,2024-02-28,SE


## Inspect customer_id field

In [18]:
cust_silver['customer_id'] = cust_silver['customer_id'].str.strip()
print("Trailing whitespaces removed from 'customer_id' column.")

Trailing whitespaces removed from 'customer_id' column.


In [19]:
empty_customer_ids = cust_silver[cust_silver['customer_id'].isnull() | (cust_silver['customer_id'] == '')]
if not empty_customer_ids.empty:
    print(f"Found {len(empty_customer_ids)} rows with empty 'customer_id' values:")
    display(empty_customer_ids)
else:
    print("No empty 'customer_id' entries found.")

duplicate_customer_ids = cust_silver[cust_silver['customer_id'].duplicated(keep=False)]
if not duplicate_customer_ids.empty:
    print(f"\nFound {len(duplicate_customer_ids)} rows with duplicate 'customer_id' values:")
    display(duplicate_customer_ids.sort_values(by='customer_id'))
    raise AssertionError("Duplicate 'customer_id' entries found. Please resolve before proceeding.")
else:
    print("No duplicate 'customer_id' entries found.")

No empty 'customer_id' entries found.

Found 2 rows with duplicate 'customer_id' values:


,customer_id,signup_date,country
37,C038,2024-12-28,NL
38,C038,2024-12-29,DE


AssertionError: Duplicate 'customer_id' entries found. Please resolve before proceeding.

### Resolving Duplicate Customer IDs

It is critical to ensure data integrity before proceeding with any analysis, especially when dealing with unique identifiers like `customer_id`. The previous check identified duplicate entries for `customer_id` **C038**.

**Action Required:**
1.  The following code cell will remove the duplicate entry for `C038` from the `cust_silver` DataFrame, keeping only the first occurrence.
2.  **Crucially, you must also ensure that customer `C038` is handled consistently in the `subscriptions` data source.** If `C038` also appears as a duplicate or has inconsistent entries in the subscriptions data, it must be resolved there as well (e.g., by removing or merging records) before any metrics are calculated or reports are generated. In a real-world scenario, you might need to investigate the source of this duplicate and apply a robust data governance strategy.

Failure to address these inconsistencies across all related datasets can lead to inaccurate metrics and flawed business insights.

In [20]:
# Remove duplicate customer_ids from cust_silver, keeping the first occurrence
original_rows = len(cust_silver)
cust_silver.drop_duplicates(subset=['customer_id'], keep='first', inplace=True)
removed_rows = original_rows - len(cust_silver)

print(f"Removed {removed_rows} duplicate customer entries from 'cust_silver'.")
print(f"'cust_silver' now has {len(cust_silver)} unique customer_id entries.")

# Re-verify for duplicates after removal
duplicate_customer_ids_after_removal = cust_silver[cust_silver['customer_id'].duplicated(keep=False)]
if not duplicate_customer_ids_after_removal.empty:
    print("Warning: Duplicate 'customer_id' entries still found after attempted removal.")
    display(duplicate_customer_ids_after_removal)
else:
    print("Verification successful: No duplicate 'customer_id' entries found in 'cust_silver' after cleanup.")

Removed 1 duplicate customer entries from 'cust_silver'.
'cust_silver' now has 39 unique customer_id entries.
Verification successful: No duplicate 'customer_id' entries found in 'cust_silver' after cleanup.


## Inspect signup_date field

In [21]:
cust_silver['signup_date'] = cust_silver['signup_date'].str.strip()
print("Trailing whitespaces removed from 'signup_date' column.")

Trailing whitespaces removed from 'signup_date' column.


In [22]:
print("\n--- Inspecting 'signup_date' field ---")

# Check for empty entries (null or empty strings)
empty_signup_dates = cust_silver[cust_silver['signup_date'].isnull() | (cust_silver['signup_date'] == '')]
if not empty_signup_dates.empty:
    print(f"Found {len(empty_signup_dates)} rows with empty 'signup_date' values:")
    display(empty_signup_dates)
else:
    print("No empty 'signup_date' entries found.")

# Convert 'signup_date' to datetime and identify malformed dates
cust_silver['signup_date_parsed'] = pd.to_datetime(cust_silver['signup_date'], errors='coerce')

malformed_dates = cust_silver[cust_silver['signup_date_parsed'].isnull() & cust_silver['signup_date'].notnull()]

if not malformed_dates.empty:
    print(f"\nFound {len(malformed_dates)} rows with malformed 'signup_date' values:")
    display(malformed_dates[['customer_id', 'signup_date']])
    raise ValueError("Malformed 'signup_date' entries found. Please resolve before proceeding.")
else:
    print("No malformed 'signup_date' entries found. All dates are valid.")

# Drop the original string column and rename the parsed one
cust_silver.drop('signup_date', axis=1, inplace=True)
cust_silver.rename(columns={'signup_date_parsed': 'signup_date'}, inplace=True)
print("Original 'signup_date' column replaced with parsed datetime objects.")


--- Inspecting 'signup_date' field ---
No empty 'signup_date' entries found.

Found 1 rows with malformed 'signup_date' values:


,customer_id,signup_date
18,C019,2024-13-05


ValueError: Malformed 'signup_date' entries found. Please resolve before proceeding.

### Resolving Malformed Signup Date

A malformed `signup_date` entry has been identified for `customer_id` **C019** ('2024-13-05'). Since this date is clearly invalid and cannot be automatically corrected, it must be removed to ensure data quality.

**Action Required:**
1.  The following code cell will remove the row corresponding to `customer_id` **C019** from the `cust_silver` DataFrame.
2.  **Crucially, you must also ensure that customer `C019` is handled consistently in the `subscriptions` data source.** If `C019` exists in the subscriptions data, it must be excluded or removed from any metrics calculations to prevent inaccuracies. In a real-world scenario, you might investigate the source of such invalid data and apply a more robust data validation and cleaning strategy.

Failure to address these inconsistencies across all related datasets can lead to inaccurate metrics and flawed business insights.

In [23]:
# Identify the customer with the malformed signup_date
malformed_customer_id = 'C019'

# Remove the row for the identified customer from cust_silver
initial_rows_count = len(cust_silver)
cust_silver = cust_silver[cust_silver['customer_id'] != malformed_customer_id].copy()
removed_rows_count = initial_rows_count - len(cust_silver)

print(f"Removed {removed_rows_count} row(s) with malformed 'signup_date' for customer_id '{malformed_customer_id}'.")
print(f"'cust_silver' now has {len(cust_silver)} rows.")

# Re-verify that the customer is no longer in the DataFrame
if malformed_customer_id in cust_silver['customer_id'].values:
    print(f"Warning: Customer '{malformed_customer_id}' still found in 'cust_silver' after attempted removal.")
else:
    print(f"Verification successful: Customer '{malformed_customer_id}' has been removed from 'cust_silver'.")

# Ensure 'signup_date' is now datetime type and no malformed dates remain
cust_silver['signup_date_parsed'] = pd.to_datetime(cust_silver['signup_date'], errors='coerce')
remaining_malformed = cust_silver[cust_silver['signup_date_parsed'].isnull() & cust_silver['signup_date'].notnull()]

if not remaining_malformed.empty:
    print("Warning: Remaining malformed 'signup_date' entries found after cleanup.")
    display(remaining_malformed[['customer_id', 'signup_date']])
else:
    print("All 'signup_date' entries are now successfully parsed as datetime objects or removed.")

# Finalizing the signup_date column
cust_silver.drop('signup_date', axis=1, inplace=True)
cust_silver.rename(columns={'signup_date_parsed': 'signup_date'}, inplace=True)
print("Original 'signup_date' column replaced with parsed datetime objects, and malformed entries removed.")

Removed 1 row(s) with malformed 'signup_date' for customer_id 'C019'.
'cust_silver' now has 38 rows.
Verification successful: Customer 'C019' has been removed from 'cust_silver'.
All 'signup_date' entries are now successfully parsed as datetime objects or removed.
Original 'signup_date' column replaced with parsed datetime objects, and malformed entries removed.
